# Part B - Q2: Fine-tuning VGG19 for 15-class Classification

**Task:** Fine-tune the `vgg19` model on this dataset, train a 15-class classification
model for **3 epochs**, and report **per-class precision and recall**.

**Difference from Q1:** In Q1 the backbone was frozen (feature extraction). Here we
**fine-tune the whole network end-to-end** (all convolutional + classifier layers are
trainable) with a small learning rate, so the pretrained features adapt to this dataset.

In [1]:
import os, random, numpy as np, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, precision_recall_fscore_support
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_ROOT = os.path.join("data", "classification", "dataset")
classes = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
cls2idx = {c: i for i, c in enumerate(classes)}
print(f"{len(classes)} classes:", classes)

Using device: cuda
15 classes: ['accordion', 'bass', 'camera', 'crocodile', 'crocodile_head', 'cup', 'dollar_bill', 'emu', 'gramophone', 'hedgehog', 'nautilus', 'pizza', 'pyramid', 'sea_horse', 'windsor_chair']


In [2]:
def build_split(root):
    train_items, test_items = [], []
    for c in classes:
        files = sorted(f for f in os.listdir(os.path.join(root, c))
                       if f.lower().endswith((".jpg", ".jpeg", ".png")))
        for f in files:
            num = int("".join(ch for ch in os.path.splitext(f)[0] if ch.isdigit()))
            path = os.path.join(root, c, f)
            (train_items if num <= 40 else test_items).append((path, cls2idx[c]))
    return train_items, test_items

train_items, test_items = build_split(DATA_ROOT)
print(f"Train images: {len(train_items)}  |  Test images: {len(test_items)}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImageList(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return self.tf(Image.open(path).convert("RGB")), label

train_loader = DataLoader(ImageList(train_items, train_tf), batch_size=16, shuffle=True, num_workers=0)
test_loader = DataLoader(ImageList(test_items, eval_tf), batch_size=16, shuffle=False, num_workers=0)

Train images: 600  |  Test images: 205


## Build VGG19 with ALL layers trainable (full fine-tuning)

In [3]:
model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(classes))
for p in model.parameters():           # ensure every layer is trainable
    p.requires_grad = True
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} (entire network is fine-tuned)")

# small LR is important for full fine-tuning so pretrained weights are not destroyed
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

Trainable parameters: 139,631,695 (entire network is fine-tuned)


In [4]:
EPOCHS = 3
for epoch in range(1, EPOCHS + 1):
    model.train()
    running, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    print(f"epoch {epoch}/{EPOCHS}  loss={running/total:.4f}  train_acc={correct/total:.4f}")

epoch 1/3  loss=1.1096  train_acc=0.6750


epoch 2/3  loss=0.1751  train_acc=0.9467


epoch 3/3  loss=0.0735  train_acc=0.9783


## Evaluate: per-class precision and recall on the test set

In [5]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds = model(x.to(device)).argmax(1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(y.numpy().tolist())

print("Fine-tuned VGG19 - per-class precision / recall (test set):")
print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))

p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(classes))), zero_division=0)
df = pd.DataFrame({"precision": p.round(3), "recall": r.round(3), "f1": f.round(3), "support": s}, index=classes)
print(df)
print(f"\nOverall test accuracy: {np.mean(np.array(y_true)==np.array(y_pred)):.4f}")

Fine-tuned VGG19 - per-class precision / recall (test set):
                precision    recall  f1-score   support

     accordion      0.938     1.000     0.968        15
          bass      0.824     1.000     0.903        14
        camera      0.909     1.000     0.952        10
     crocodile      0.750     0.900     0.818        10
crocodile_head      1.000     0.818     0.900        11
           cup      1.000     0.941     0.970        17
   dollar_bill      1.000     1.000     1.000        12
           emu      1.000     1.000     1.000        13
    gramophone      1.000     0.818     0.900        11
      hedgehog      1.000     0.929     0.963        14
      nautilus      0.938     1.000     0.968        15
         pizza      1.000     1.000     1.000        13
       pyramid      1.000     0.941     0.970        17
     sea_horse      1.000     0.941     0.970        17
 windsor_chair      1.000     1.000     1.000        16

      accuracy                          0.

## Observations
* Full fine-tuning lets the convolutional filters of VGG19 adapt to this dataset, so it
  usually matches or beats the frozen-backbone VGG19 from Q1, especially on the harder,
  visually similar classes (`crocodile` vs `crocodile_head`, `sea_horse`).
* Because we only have 40 training images per class and train for just 3 epochs, a small
  learning rate (SGD 1e-3) is used to avoid over-writing the pretrained ImageNet features
  and over-fitting.
* Per-class recall is the most informative metric here: classes with cleaner, more
  distinctive appearance reach precision/recall ~1.0, while the confusable pairs remain
  the main error source.